# Подготовка BIO-разметки для NER

В EDA мы выяснили, что товар и бренд почти всегда присутствуют в тексте позиции
как подстрока. Значит можем разметить их по схеме BIO и обучать
token-classification.

Здесь мы превращаем строковую разметку (текст + нормализованные good и brand) в
токенную: каждому слову позиции ставим тег - B-GOOD, I-GOOD, B-BRAND, I-BRAND или
O. Размечаем на уровне слов (split по пробелам); выравнивание по сабтокенам
xlm-roberta сделаем позже, при обучении.

Правила, к которым пришли:
- позицию берём в обучение, если удалось разметить хотя бы товар;
- если бренда нет или он не нашёлся это валидный случай без бренда, оставляем;
- метку ищем без учёта регистра, по границам слов.

## Запуск в Google Colab

Ноутбук рассчитан на запуск в Colab по кнопке **Run all** без ручных действий:
код клонируется из GitHub, датасет OFD (`train_supervised.csv`) скачивается с
Google Drive. Размеченные BIO-файлы (`data/processed/ofd_ner/`) генерируются
здесь же детерминированно (seed=42).


In [ ]:
# === Настройка окружения для Colab (локально ничего не делает) ===
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Публичный git-репозиторий с кодом проекта.
REPO_URL = "https://github.com/ScarletFlame611/Receipt-AI.git"
# id файла train_supervised.csv на Google Drive (тот же, что в ноутбуке 06).
OFD_GDRIVE_ID = "1VjvhgFEwguDDc_rE2B-q82tfyoxRnTYl"


def _find_project_root():
    """Корень проекта — папка, где есть src/. Текущая папка или склонированная рядом."""
    here = Path.cwd()
    if (here / "src").is_dir():
        return here
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "src").is_dir():
            return sub
    return None


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "pandas", "numpy", "gdown"],
        check=False,
    )
    if _find_project_root() is None and "USER/" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL], check=True)

root = _find_project_root()
if root is None:
    raise RuntimeError(
        "Не найден код проекта (папка src/). В Colab склонируй репозиторий "
        "(или задай REPO_URL) и перезапусти ячейку."
    )
os.chdir(root)
sys.path.insert(0, str(root))
print("Colab:", IN_COLAB, "| корень проекта:", root)


def ensure_ofd_csv(root):
    """Датасет OFD (data/raw/ofd/train_supervised.csv). Нет локально -> качаем с Google Drive."""
    dst = Path(root) / "data" / "raw" / "ofd" / "train_supervised.csv"
    if dst.exists():
        return dst
    import gdown
    dst.parent.mkdir(parents=True, exist_ok=True)
    print("OFD CSV не найден локально — качаю с Google Drive...")
    gdown.download(id=OFD_GDRIVE_ID, output=str(dst), quiet=False)
    return dst


In [1]:
import sys
from pathlib import Path
import re
import pandas as pd
import numpy as np

def find_root(start):
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() or (cand / "data").exists():
            return cand
    return p

root = find_root(Path.cwd())
sys.path.insert(0, str(root))

ensure_ofd_csv(root)  # нет локально -> качаем CSV с Google Drive
df = pd.read_csv(root / "data" / "raw" / "ofd" / "train_supervised.csv")
print("Загружено:", df.shape)

Загружено: (25000, 4)


## Генератор разметки

Размечаем по словам. Для позиции берём текст, бьём на слова, и для good и brand
ищем, где их слова идут подряд в тексте - это вхождение помечаем тегами B-/I-.
Сравниваем в нижнем регистре. Если сущность из нескольких слов (например бренд
«коровка из кореновки»), первое слово получает B-, остальные I-.

Товар размечаем первым, бренд вторым. Если они пересеклись (одно слово попало и
туда, и туда), приоритет у бренда т.к. он обычно конкретнее.

In [5]:
LABELS = ["O", "B-GOOD", "I-GOOD", "B-BRAND", "I-BRAND"]

_token_re = re.compile(r"[A-Za-zА-Яа-яЁё]+|\d+|[^\sA-Za-zА-Яа-яЁё\d]", re.UNICODE)

def normalize(text):
    return str(text).replace("`", "'").replace("’", "'").replace("´", "'")

def tokenize_simple(text):
    return _token_re.findall(normalize(text))

def find_span(words_lower, entity):
    ent_words = tokenize_simple(str(entity).lower())
    ent_words = [w for w in ent_words if w]
    if not ent_words:
        return None
    n = len(ent_words)
    for i in range(len(words_lower) - n + 1):
        if words_lower[i:i+n] == ent_words:
            return (i, i + n)
    return None

def label_position(name, good, brand):
    words = tokenize_simple(name)
    words_lower = [w.lower() for w in words]
    tags = ["O"] * len(words)

    good_ok = False
    if pd.notna(good):
        for ent in str(good).split(","):
            span = find_span(words_lower, ent.strip())
            if span:
                s, e = span
                tags[s] = "B-GOOD"
                for j in range(s+1, e):
                    tags[j] = "I-GOOD"
                good_ok = True

    if pd.notna(brand):
        for ent in str(brand).split(","):
            span = find_span(words_lower, ent.strip())
            if span:
                s, e = span
                tags[s] = "B-BRAND"
                for j in range(s+1, e):
                    tags[j] = "I-BRAND"

    return words, tags, good_ok

In [6]:
good_ok_count = 0
brand_found_count = 0
brand_total = df["brand"].notna().sum()

for _, r in df.iterrows():
    words, tags, good_ok = label_position(r["name"], r["good"], r["brand"])
    if good_ok:
        good_ok_count += 1
    if "B-BRAND" in tags:
        brand_found_count += 1

print(f"Товар размечен: {good_ok_count}/{len(df)} = {good_ok_count/len(df):.1%}")
print(f"Бренд размечен: {brand_found_count}/{brand_total} = {brand_found_count/brand_total:.1%}")

Товар размечен: 22501/25000 = 90.0%
Бренд размечен: 14031/16495 = 85.1%


### Покрытие после умной токенизации

После того как стали резать текст не только по пробелам, но и по разделителям
(/, +, _, точки) и нормализовать апострофы, покрытие выросло до 90% по товарам и
85% по брендам. То есть из
найденных подстрок мы теряем при разметке почти ничего.

Остаток (10% товаров) это в основном развёрнутые сокращения, которых в тексте
дословно нет вообще. Такие
позиции либо разметятся только по бренду, либо уйдут из обучения, если в них не
нашёлся даже товар.

## Разметка всего датасета и сохранение

Размечаем все позиции, оставляем те, где удалось разметить хотя бы товар.
Сохраняем в JSONL — по строке на позицию со списком слов и списком тегов.
Этот формат напрямую читается HuggingFace datasets в обучающем ноутбуке.

In [7]:
import json

records = []
dropped = 0
for _, r in df.iterrows():
    words, tags, good_ok = label_position(r["name"], r["good"], r["brand"])
    has_brand = "B-BRAND" in tags
    if not good_ok and not has_brand:
        dropped += 1
        continue
    records.append({"tokens": words, "ner_tags": tags})

print("Размечено позиций:", len(records))
print("Отброшено (ни товара, ни бренда):", dropped)

tag_counter = {}
for rec in records:
    for t in rec["ner_tags"]:
        tag_counter[t] = tag_counter.get(t, 0) + 1
print("\nРаспределение тегов:")
for t in LABELS:
    print(f"  {t}: {tag_counter.get(t, 0)}")

Размечено позиций: 23847
Отброшено (ни товара, ни бренда): 1153

Распределение тегов:
  O: 212242
  B-GOOD: 22539
  I-GOOD: 596
  B-BRAND: 14031
  I-BRAND: 2985


### Что получилось

В обучение идёт 23847 позиций, отбросили 1153 (там не нашлось ни товара, ни
бренда и размечать нечего). Это меньше 5%, потеря приемлемая.

По тегам видно ожидаемую для NER картину: O сильно доминирует. Это нормально т.к. большая часть слов в позиции это не товар и не
бренд, а описания, объёмы, коды. Модель с таким дисбалансом справляется, специально
бороться обычно не нужно, но при оценке смотрим не на accuracy (она будет высокой
просто за счёт O), а на F1 по GOOD и BRAND отдельно.

Ещё заметно, что товары почти всегда однословные,
а бренды чаще бывают многословными. Логика разметки их обработала.

## Train/val/test и сохранение в JSONL

Делим 80/10/10 с фиксированным seed. Сохраняем три файла в data/processed/ofd_ner/.
Эти файлы заливаются на Google Drive, оттуда их грузит обучающий ноутбук в Colab.

In [11]:
import random

random.Random(42).shuffle(records)
n = len(records)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

splits = {
    "train": records[:n_train],
    "validation": records[n_train:n_train + n_val],
    "test": records[n_train + n_val:],
}

out_dir = root / "data" / "processed" / "ofd_ner"
out_dir.mkdir(parents=True, exist_ok=True)

for split_name, recs in splits.items():
    path = out_dir / f"{split_name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"{split_name}: {len(recs)}  {path}")


train: 19077  C:\Users\kseni\PycharmProjects\Receipt-AI\data\processed\ofd_ner\train.jsonl
validation: 2384  C:\Users\kseni\PycharmProjects\Receipt-AI\data\processed\ofd_ner\validation.jsonl
test: 2386  C:\Users\kseni\PycharmProjects\Receipt-AI\data\processed\ofd_ner\test.jsonl
